# 00 · 本地环境与模型服务
**请在上课前完成。** 本 notebook 可以独立运行，不包含实践作业的答案。


**第一次打开时：** 安装 Python 3.12，将本 notebook 保存到可写文件夹中，然后在**终端**（而不是 Python 单元格）里运行以下命令。如果你下载了学生资料包，请先解压，并在解压后的文件夹中操作。

| 平台 | 终端命令（每次运行一条） |
|---|---|
| Windows | `py -3.12 -m venv .venv`<br>`.\.venv\Scripts\python.exe -m pip install "jupyterlab>=4,<5" "ipywidgets>=8.1,<9" "ipykernel>=6,<8"`<br>`.\.venv\Scripts\python.exe -m jupyterlab` |
| macOS / Linux | `python3 -m venv .venv`<br>`.venv/bin/python -m pip install 'jupyterlab>=4,<5' 'ipywidgets>=8.1,<9' 'ipykernel>=6,<8'`<br>`.venv/bin/python -m jupyterlab` |

在浏览器中打开本 notebook，并选择这个环境中的 Python 内核。如果现有课程环境能通过下面的检查，也可以继续使用。不需要 GPU 或付费 API。请使用本地 Jupyter 内核：托管 notebook 中的 `localhost` 并不是你的电脑。

## 1 · Python 与 notebook 依赖项
请从上到下依次运行。环境检查不会安装或替换 Python。如果内核选错了，必须在 Jupyter 中切换。

In [ ]:
import sys, json, platform, importlib.util, sqlite3, subprocess, shutil, os, time
from pathlib import Path
from urllib import request, error
from urllib.parse import urlparse

ROOT = Path.cwd().resolve()
if sys.version_info < (3, 10):
    raise RuntimeError("需要 Python 3.10 或更高版本；课程环境请使用 Python 3.12。")
print("Python 版本：", sys.version.split()[0], "| 可执行文件：", sys.executable)
print("工作文件夹：", ROOT)
print("SQLite:", sqlite3.sqlite_version)
with sqlite3.connect(":memory:") as conn:
    assert conn.execute("SELECT 1").fetchone()[0] == 1

In [ ]:
# 仅当当前内核缺少 widget 依赖项时，才将其安装到这个内核中。
if importlib.util.find_spec("ipywidgets") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "ipywidgets>=8.1,<9"], check=True)
    print("如果下面的 widget 无法显示，请重启内核并再次运行。")
import ipywidgets as widgets
from IPython.display import display
widget_test = widgets.IntSlider(description="Widget 测试")
display(widget_test)
print("请移动滑块。若只显示纯文本形式，说明 widget 界面没有正常工作。")

## 2 · 安装一次 Ollama
Ollama 是一个独立应用程序；仅运行 `pip install ollama` 并不会安装 Ollama 服务器。

- **Windows：** 使用 [ollama.com/download/windows](https://ollama.com/download/windows) 提供的安装程序，然后从开始菜单启动 Ollama。
- **macOS：** 使用 [ollama.com/download/mac](https://ollama.com/download/mac) 提供的安装程序，然后打开已安装的应用程序。
- **Linux：** 按照[官方安装页面](https://docs.ollama.com/linux)操作。在终端中运行安装命令 `curl -fsSL https://ollama.com/install.sh | sh`。如果服务尚未运行，可用 `ollama serve` 启动服务器。

本 notebook 会先检查服务器，再尝试启动服务器。桌面版通常已经在运行服务。安装完成后，如果 Jupyter 的终端找不到新的 `ollama` 命令，请重启 Jupyter。

官方平台要求：[Windows](https://docs.ollama.com/windows)、[macOS](https://docs.ollama.com/macos)、[Linux](https://docs.ollama.com/linux)。请选择**本地模型**；本练习不需要云端账号或 API 密钥。

In [ ]:
# 保留这个本地端点，后续应用程序会读取相同的配置。
BASE_URL = "http://127.0.0.1:11434"
MODEL_NAME = "llama3.2:1b"
CPU_ONLY = True
THINK = None 
START_SERVER_IF_NEEDED = True
DOWNLOAD_IF_MISSING = True

parsed = urlparse(BASE_URL)
assert parsed.scheme == "http" and parsed.hostname in {"localhost", "127.0.0.1", "::1"}
assert not MODEL_NAME.endswith(":cloud"), "请选择本地模型标签。"

# 重新运行此单元格时，仍会保留对本 notebook 所启动服务器的引用。
if "started_server" not in globals():
    started_server = None

def api_json(path, payload=None, timeout=5):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = request.Request(BASE_URL.rstrip("/") + path, data=data,
                          headers={"Content-Type": "application/json"})
    try:
        with request.urlopen(req, timeout=timeout) as response:
            return json.load(response)
    except error.HTTPError as exc:
        detail = exc.read(2000).decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama HTTP {exc.code}: {detail}") from exc

def server_available():
    try:
        return isinstance(api_json("/api/version").get("version"), str)
    except (OSError, RuntimeError, ValueError):
        return False

In [ ]:
if not server_available():
    cli = shutil.which("ollama")
    if not START_SERVER_IF_NEEDED or cli is None:
        raise RuntimeError("请打开或安装 Ollama，或者在终端运行 'ollama serve'，然后重新运行此单元格。")
    server_env = os.environ.copy()
    server_env["OLLAMA_HOST"] = parsed.netloc
    # 将日志输出到文件，避免 'serve' 让这个 notebook 单元格一直阻塞。
    with (ROOT / "ollama_setup.log").open("ab") as log:
        started_server = subprocess.Popen([cli, "serve"], stdout=log,
                                          stderr=subprocess.STDOUT, env=server_env)
    for _ in range(20):
        if server_available():
            break
        if started_server.poll() is not None:
            break
        time.sleep(0.5)
if not server_available():
    raise RuntimeError("服务器未能就绪。请检查 ollama_setup.log 或 Ollama 应用程序日志。")
server_version = api_json("/api/version")["version"]
print("Ollama 服务器：", server_version, "地址：", BASE_URL)
print("是否由本 notebook 启动：", started_server is not None and started_server.poll() is None)

## 3 · 上课前下载一个小型模型
默认的 [Llama3.2:1b](https://ollama.com/library/llama3.2:1b) 是用于分类练习的小型基准模型，并不保证回答质量。请保留教师指定的模型标签，以便比较结果。无需下载更新或更大的模型。

下一个单元格会列出本地模型，并且只在缺少该标签时下载模型。首次下载需要网络连接和磁盘空间，可能需要几分钟，请让该单元格继续运行。也可以在终端运行 `ollama pull llama3.2:1b`。

模型文件由 Ollama 存储在 notebook 资料包之外。重启 Jupyter 后，这些文件仍然可用。模型占用的磁盘空间不等于推理时占用的内存。

In [ ]:
models = api_json("/api/tags").get("models", [])
installed_names = {m.get("name") for m in models}
print("已安装：", sorted(n for n in installed_names if n))
if MODEL_NAME not in installed_names:
    if not DOWNLOAD_IF_MISSING:
        raise RuntimeError(f"请使用 'ollama pull {MODEL_NAME}' 下载 {MODEL_NAME}，然后重新运行。")
    req = request.Request(BASE_URL.rstrip("/") + "/api/pull",
        data=json.dumps({"model": MODEL_NAME, "stream": True}).encode("utf-8"),
        headers={"Content-Type": "application/json"})
    last_progress = None
    # NDJSON 进度流；超时时间只适用于套接字 I/O，并不限制整个下载过程。
    with request.urlopen(req, timeout=120) as response:
        for line in response:
            if not line.strip():
                continue
            event = json.loads(line)
            if event.get("error"):
                raise RuntimeError(event["error"])
            percent = int(100 * event.get("completed", 0) / event["total"]) if event.get("total") else None
            progress = (event.get("status"), None if percent is None else percent // 10)
            if progress != last_progress:
                print(event.get("status"), "" if percent is None else f"{percent}%", flush=True)
                last_progress = progress
models = api_json("/api/tags").get("models", [])
selected = next((m for m in models if m.get("name") == MODEL_NAME), None)
if selected is None:
    raise RuntimeError("仍然找不到指定的模型标签。请检查 /api/tags 或运行 'ollama list'。")
print("磁盘中的模型已就绪：", selected["name"], "| 摘要：", selected.get("digest"))

## 4 · 真实的结构化输出冒烟测试
这里会向你的 Ollama 服务器发送一次 HTTP 请求，用来测试模型服务、schema 支持以及响应是否完整，而不是测试客服任务的准确度。

CPU 设置会发送 `num_gpu=0`。模型加载后，请检查进程信息（或运行 `ollama ps`），确认模型实际运行的位置。第一次调用可能包含模型加载时间。配置的套接字超时时间为 120 秒，这并不代表响应一定会在该时间内完成。

In [ ]:
probe_schema = {"type":"object", "properties":{"ready":{"type":"boolean"}},
                "required":["ready"], "additionalProperties":False}
config = {"model":MODEL_NAME, "base_url":BASE_URL, "timeout_s":120,
          "output_mode":"schema", "num_ctx":2048, "num_predict":96,
          "cpu_only":CPU_ONLY, "think":THINK}
options = {"temperature":0, "num_ctx":config["num_ctx"], "num_predict":config["num_predict"]}
if CPU_ONLY:
    options["num_gpu"] = 0
payload = {"model":MODEL_NAME, "messages":[{"role":"user", "content":'Return only {"ready":true}.'}],
           "stream":False, "format":probe_schema, "options":options, "keep_alive":"5m"}
if THINK is not None:
    payload["think"] = THINK
start = time.perf_counter()
probe = api_json("/api/chat", payload, timeout=config["timeout_s"])
probe_elapsed_s = time.perf_counter() - start
if probe.get("done") is not True or probe.get("done_reason") == "length":
    raise RuntimeError("模型输出不完整。请检查模型、思考设置和输出长度限制。")
probe_value = json.loads(probe["message"]["content"])
if probe_value != {"ready":True} or type(probe_value["ready"]) is not bool:
    raise RuntimeError(f"测试输出不符合预期：{probe_value!r}")
print("结构化输出：", probe_value)
print("耗时（秒）：", round(probe_elapsed_s, 3))
print("Token 数量：", {k:probe.get(k) for k in ("prompt_eval_count","eval_count")})

In [ ]:
processes = api_json("/api/ps").get("models", [])
loaded = next((m for m in processes if m.get("name") == MODEL_NAME or m.get("model") == MODEL_NAME), None)
cpu_verified = bool(loaded is not None and loaded.get("size_vram") == 0)
print("已加载的进程：", loaded)
if CPU_ONLY and not cpu_verified:
    print("尚未确认模型在 CPU 上运行。请检查 'ollama ps'，并向教学团队报告。")
else:
    print("已确认模型在 CPU 上运行。" if CPU_ONLY else "已记录硬件运行位置。")

## 5 · 保存配置和就绪报告
如果以后将实践作业解压到其他位置，请将这两个文件与 notebook 一起移动。实践作业会从根文件夹读取 `agentic_ai_config.json`。请将模型名称、配置和测量结果保存在一起。

滑块是否正常工作需要人工确认：只有成功移动滑块后，才可以修改下面的值。测试通过并不能证明模型准确度达到要求，也不能证明其符合课堂上的延迟目标。

In [ ]:
WIDGET_DISPLAY_CONFIRMED = False  # 成功移动第 1 节中的滑块后，将此值改为 True。

(ROOT / "agentic_ai_config.json").write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
report = {"python":sys.version.split()[0], "platform":platform.system(),
          "sqlite_version":sqlite3.sqlite_version, "ollama_version":server_version,
          "model":MODEL_NAME, "model_digest":selected.get("digest"),
          "model_size_bytes":selected.get("size"), "config":config,
          "schema_probe_passed":True, "probe_elapsed_s":round(probe_elapsed_s,6),
          "cpu_placement_verified":cpu_verified, "widget_display_confirmed":WIDGET_DISPLAY_CONFIRMED}
report["ready_for_practical"] = bool(WIDGET_DISPLAY_CONFIRMED and (not CPU_ONLY or cpu_verified))
(ROOT / "setup_report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print("已保存配置和 setup_report.json")
print("是否已为实践作业做好准备：", report["ready_for_practical"])
print("如果结果为 False，请在上课前解决报告指出的问题；如有需要，请将报告发送给教学团队。")

## 故障排查

| 症状 | 解决方法 |
|---|---|
| 没有 Jupyter 命令或 Python 版本错误 | 按照顶部说明，使用该环境中 Python 的完整路径启动。 |
| Widget 显示为文本 | 检查当前内核中的 `ipywidgets` 和 JupyterLab 环境中的 `jupyterlab_widgets`，然后重启内核或浏览器。 |
| 连接被拒绝 | 打开 Ollama，或运行 `ollama serve`；确保内核运行在同一台电脑上。 |
| 地址已被占用 | 服务器可能已经在运行。检查 `/api/version`，不要启动第二个实例。 |
| 找不到模型 | 将准确的 `MODEL_NAME` 与 `/api/tags` 或 `ollama list` 的结果进行比较。 |
| 下载中断 | 恢复网络连接后，重新运行下载单元格或 `ollama pull`。 |
| 操作系统不受支持或安装被阻止 | 检查官方平台要求，并在上课前联系教学团队。 |
| 第一次调用很慢 | 分别记录模型加载时间和预热后的调用时间，并关闭其他占用资源较多的进程。 |
| 输出为空或被截断 | 保留默认的非思考模型，或明确关闭模型支持的思考输出；同时检查输出长度限制。 |
| Schema 被拒绝 | 更新 Ollama 或报告其版本；不要把仅由提示词约束的输出称为受 schema 约束的输出。 |
| 无法确定是否使用 CPU | 检查 `ollama ps`，并提供模型标签、服务器版本和设置报告。 |

第二天继续时：启动 Ollama，从同一个环境启动 Jupyter，然后打开 `01_Practical_1_Student.ipynb`。已经存在的模型不需要重新下载。

In [ ]:
# 可选清理：只停止由本 notebook 启动的服务器。
# 进行实践作业时请保持 False；桌面或系统服务不会受到影响。
STOP_NOTEBOOK_SERVER = False
if STOP_NOTEBOOK_SERVER and started_server is not None and started_server.poll() is None:
    started_server.terminate()
    started_server.wait(timeout=10)
    print("已停止由本 notebook 启动的服务器。需要时请重新运行第 2 节。")

## 主要参考资料
核对日期：2026 年 9 月 24 日。[Ollama 安装与本地服务](https://docs.ollama.com/quickstart)、[聊天 API](https://docs.ollama.com/api/chat)、[结构化输出](https://docs.ollama.com/capabilities/structured-outputs)、[模型列表](https://docs.ollama.com/api/tags)、[模型下载](https://docs.ollama.com/api/pull)、[运行时选项源码](https://github.com/ollama/ollama/blob/main/api/types.go)、[Python 环境](https://docs.python.org/3/library/venv.html)、[JupyterLab 安装](https://jupyterlab.readthedocs.io/en/stable/getting_started/installation.html)。